### Подготовка данных


In [6]:
import os
import shutil
from pathlib import Path
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import joblib

In [7]:
home = '/home/slava/Documents/netology_ML/Diplom'

diff_spot_actual_bunner_dir = home + '/diff_spot_bunner_actual' # скриншоты diff spot
diff_spot_actual_dir = home + '/diff_spot_actual' # скриншоты diff spot

In [3]:
data = []

def create_dataset(path_actual, label):
    for filename in os.listdir(path_actual):
        actual_path = os.path.join(path_actual, filename)
        expected_path = os.path.join(path_actual.replace('actual', 'expected'), filename.replace('aug_', ''))
        if os.path.isfile(actual_path) and os.path.isfile(expected_path):
            data.append({
                'actual': str(actual_path),
                'expected': str(expected_path),
                'filename': str(filename),
                'label': label
            })
        else:
            print(f"Пропущен файл {filename}: один из файлов не существует")

create_dataset(diff_spot_actual_bunner_dir, 0)
create_dataset(diff_spot_actual_dir, 1)
                
df = pd.DataFrame(data)
df.to_csv('labels_6_overeducation4.csv', index=False)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7548 entries, 0 to 7547
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   actual    7548 non-null   object
 1   expected  7548 non-null   object
 2   filename  7548 non-null   object
 3   label     7548 non-null   int64 
dtypes: int64(1), object(3)
memory usage: 236.0+ KB


In [5]:
IMG_SIZE = (512, 256)  # до какого размера сжимаем изображения

def compress_image(path):
    img = Image.open(path).convert('L')  # конвертируем в grayscale
    img = img.resize(IMG_SIZE)  # сжимаем
    return np.array(img) / 255.0  # нормализация

X = []
y = []
actual_images = []
expected_images = []

for _, row in df.iterrows():
    actual = compress_image(row['actual'])
    expected = compress_image(row['expected'])
    actual_images.append(actual)
    expected_images.append(expected)

    # Вычисляем абсолютную разницу (в градациях серого)
    diff = np.abs(actual - expected)  # форма: (h, w)
    
    X.append(diff)
    y.append(row['label'])

joblib.dump(actual_images, 'actual_images_after_compress_6_overeducation4.joblib')
joblib.dump(expected_images, 'expected_images_after_compress_6_overeducation4.joblib')
joblib.dump(X, 'diff_images_after_compress_6_overeducation4.joblib')
joblib.dump(y, 'y_labels_6_overeducation4.joblib')

X = np.array(X)  # вертор разницы
y = np.array(y)  # метки: 0 или 1

In [3]:
X = joblib.load('diff_images_after_compress_6_overeducation4.joblib')
y = joblib.load('y_labels_6_overeducation4.joblib')
X = np.array(X)  # вертор разницы
y = np.array(y)  # метки: 0 или 1

print(f"Форма X: {X.shape}")
print(f"Тип данных X: {X.dtype}")
print(f"Количество образцов: {len(X)}")
print(f"Диапазон значений: min={X.min():.4f}, max={X.max():.4f}")
print(f"Среднее значение: {X.mean():.4f}")
print(f"Стандартное отклонение: {X.std():.4f}")

Форма X: (7548, 256, 512)
Тип данных X: float64
Количество образцов: 7548
Диапазон значений: min=0.0000, max=1.0000
Среднее значение: 0.0164
Стандартное отклонение: 0.1000


In [4]:
def crop_to_content_with_coords(image, threshold=0.01):
    """
    Обрезает изображение до области с информацией и возвращает координаты.
    
    Returns:
        cropped: обрезанное изображение
        top_left: кортеж (y, x) — координаты верхнего левого угла обрезанной области 
                  относительно исходного изображения (до применения padding)
    """
    mask = image > threshold
    
    if not mask.any():
        return np.zeros((10, 10)), (0, 0)
    
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    
    # Сохраняем координаты ДО добавления padding
    top_left_y = rmin
    top_left_x = cmin
    
    padding = 10
    rmin = max(0, rmin - padding)
    rmax = min(image.shape[0], rmax + padding)
    cmin = max(0, cmin - padding)
    cmax = min(image.shape[1], cmax + padding)
    
    cropped = image[rmin:rmax, cmin:cmax]
    top_left = (top_left_y, top_left_x)
    
    return cropped, top_left

X_cropped = []
X_coords = []
for img in X:
    cropped, coords = crop_to_content_with_coords(img)
    X_cropped.append(cropped)
    X_coords.append(coords)

# Проверяем результаты
print(f"Размеры до обрезки: {X[0].shape}")
print(f"Размеры после обрезки: {X_cropped[0].shape}")
print(f"Пример координат: {X_coords[0]}")
print(f"Средний размер после обрезки: {np.mean([img.shape for img in X_cropped])}")

Размеры до обрезки: (256, 512)
Размеры после обрезки: (106, 207)
Пример координат: (np.int64(88), np.int64(69))
Средний размер после обрезки: 122.39407790143085


In [8]:
joblib.dump(X_cropped, 'X_cropped_diff_images_6_overeducation4.joblib')
joblib.dump(X_coords, 'X_coords_diff_images_6_overeducation4.joblib')
joblib.dump(y, 'y_labels_6_overeducation4.joblib')

['y_labels_6_overeducation4.joblib']